# Run single sim + extract features + ED

This notebook mirrors `experiments/run_sim_extract_ed.py` and writes a single-row CSV.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import andes

ROOT_DIR = Path.cwd()
if (ROOT_DIR / 'experiments').is_dir() and (ROOT_DIR / 'scheduling').is_dir():
    pass
elif ROOT_DIR.name == 'experiments' and (ROOT_DIR.parent / 'scheduling').is_dir():
    ROOT_DIR = ROOT_DIR.parent

if ROOT_DIR != Path.cwd():
    os.chdir(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR))

from experiments.run_sim_extract_ed import _load_yaml, _repeat_or_validate, add_measurement_devices, _run_single_sim


In [ ]:
config_path = Path('experiments/generation.yaml')
cost_path = Path('scheduling/mtlsh_convex.yaml')
base_scale = 1.0
step_scale = 0.9

cfg = _load_yaml(config_path)
cost_cfg = _load_yaml(cost_path)
if 'ed_costs' not in cost_cfg:
    raise KeyError('Missing ed_costs in cost-config YAML.')


In [ ]:
andes.config_logger(stream_level=int(cfg.get('stream_level', 30)))
case_path = cfg['case']

output_dir = Path(cfg.get('output_dir', 'experiments'))
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / cfg.get('output_csv', 'simulation_results.csv')

plotter_cfg = cfg.get('plotter', {})
export_plotter = bool(plotter_cfg.get('export', False))
plotter_dir = None
if export_plotter:
    plotter_dir = output_dir / plotter_cfg.get('subdir', 'plotter')

rng = np.random.default_rng(int(cfg.get('seed', 42)))
row, Pg_opt, ed_cost, ed_lam = _run_single_sim(
    cfg,
    rng=rng,
    case_path=case_path,
    base_scale=float(base_scale),
    step_scale=float(step_scale),
    cost_cfg=cost_cfg['ed_costs'],
    export_plotter=export_plotter,
    plotter_dir=plotter_dir,
)

row['sim_id'] = 0
row['seed'] = int(cfg.get('seed', 42))
row['ed_cost'] = float(ed_cost)
row['ed_lambda'] = float(ed_lam) if ed_lam is not None else np.nan
for i, val in enumerate(Pg_opt, start=1):
    row[f'ed_Pg_{i}'] = float(val)

import csv
fieldnames = list(row.keys())
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow(row)

print(f'Wrote {csv_path}')
